In [ ]:
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

from RetinaFace import RetinaFace
from ArcFace import ArcFace

In [ ]:
snapshots_dir = Path("../volumes_backup/snapshots")
image_paths = list(snapshots_dir.glob("*.jpg"))
print(f"found {len(image_paths)} snapshots")

image_path = random.choice(image_paths)
print(image_path)

image = cv2.imread(str(image_path))
assert image is not None, f"failed to read {image_path}"

In [ ]:
retina = RetinaFace(ctx_id=-1)
arcface = ArcFace(ctx_id=-1)

faces = retina.detect(image)
print(f"detected {len(faces)} face(s)")

embeddings = []
for face in faces:
    aligned = retina.align(image, face.kps)
    embedding = arcface.embed(aligned)
    embeddings.append((face, aligned, embedding))
    print(f"bbox={face.bbox} det_score={face.det_score:.3f} embedding_shape={embedding.shape}")

In [ ]:
n = len(embeddings)
fig, axes = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 4))
if n == 0:
    axes = [axes]

vis = image.copy()
for face, _, _ in embeddings:
    x1, y1, x2, y2 = face.bbox.astype(int)
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)

axes[0].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
axes[0].set_title(image_path.name)
axes[0].axis("off")

for i, (face, aligned, _) in enumerate(embeddings):
    axes[i + 1].imshow(cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB))
    axes[i + 1].set_title(f"aligned face {i}")
    axes[i + 1].axis("off")

plt.show()

In [ ]:
# pairwise cosine similarity between detected faces in this image (sanity check)
for i in range(len(embeddings)):
    for j in range(i + 1, len(embeddings)):
        sim = ArcFace.cosine_similarity(embeddings[i][2], embeddings[j][2])
        print(f"face {i} vs face {j}: cosine similarity = {sim:.3f}")